# Dataset 4 – ABS 2021 Census Demographic Context

This notebook documents Dataset 4 for AgeTogether Iteration 1. It uses aggregate Australian Bureau of Statistics (ABS) Census data as **target-audience and design-context evidence**, not as individual profiling data and not for personalised recommendations.

## 1. Data Acquisition and Provenance

- **Provider:** Australian Bureau of Statistics (ABS)
- **Product:** 2021 Census General Community Profile
- **Geography:** Melbourne Local Government Area (LGA), code LGA24600
- **Census observation year:** 2021
- **Official profile URL:** https://www.abs.gov.au/census/find-census-data/community-profiles/2021/LGA24600
- **Official download URL:** https://www.abs.gov.au/census/find-census-data/community-profiles/2021/LGA24600/download/GCP_LGA24600.xlsx
- **Licence:** Creative Commons Attribution 4.0 International (CC BY 4.0), as verified from the ABS website copyright notice.
- **Attribution requirement:** derived material should state **“Based on Australian Bureau of Statistics data”**.
- **Local acquisition date:** 3 September 2026

The General Community Profile contains characteristics of persons, families and dwellings and is based on **place of usual residence**. It supports AgeTogether by providing local demographic context for an age-friendly Melbourne prototype. It does not describe individual users, prove needs or preferences, or support personalised recommendations.

The local acquisition date is not the Census observation year: the data describes 2021, while this repository copy was acquired in 2026.

In [ ]:
from pathlib import Path
import hashlib

import matplotlib.pyplot as plt
import pandas as pd
from openpyxl import load_workbook

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'data').is_dir() and (REPO_ROOT.parent / 'data').is_dir():
    REPO_ROOT = REPO_ROOT.parent
RAW_FILE = REPO_ROOT / 'data/raw/abs_2021_melbourne_lga24600_general_community_profile.xlsx'
OUTPUT_FILE = REPO_ROOT / 'data/processed/agetogether_abs_2021_demographic_insights.csv'

EXPECTED_SHA256 = 'c3ecc64c2675a7d10c25ea9f43d1fae2143e3231d26dd748db8d7f68ccb64b34'
GEOGRAPHY = 'Melbourne (LGA24600)'
CENSUS_YEAR = 2021

assert RAW_FILE.exists(), f'Missing raw file: {RAW_FILE}'
raw_checksum = hashlib.sha256(RAW_FILE.read_bytes()).hexdigest()
assert raw_checksum == EXPECTED_SHA256, 'Raw file checksum does not match the acquired source.'

workbook = load_workbook(RAW_FILE, read_only=True, data_only=True)
print(f'Raw workbook: {RAW_FILE.name}')
print(f'SHA-256: {raw_checksum}')
print(f'Worksheets: {len(workbook.sheetnames)}')
print('Selected tables: G04, G18, G23, G27')

## 2. Data Understanding

The raw workbook is retained unchanged in data/raw/. This analysis uses only four relevant General Community Profile tables:

- G04: Age by sex
- G18: Core activity need for assistance by age by sex
- G23: Voluntary work for an organisation or group by age by sex
- G27: Relationship in household by age by sex

All selected tables identify **Melbourne (LGA24600)** and are based on place of usual residence. Values are published counts unless explicitly labelled as percentages below.

ABS applies small random adjustments to Census cells to protect confidentiality. Therefore, small differences between related table totals are expected and are not treated as data errors.

In [ ]:
selected_table_shapes = {
    sheet: (workbook[sheet].max_row, workbook[sheet].max_column)
    for sheet in ['G04', 'G18', 'G23', 'G27']
}
print('Selected table shapes (rows, columns):')
for sheet, shape in selected_table_shapes.items():
    print(f'{sheet}: {shape}')

for sheet in ['G04', 'G18', 'G23', 'G27']:
    print(f'{sheet}: {workbook[sheet]["A1"].value} | {workbook[sheet]["A2"].value}')

## 3. Data Quality Checks

These checks validate only the selected aggregate values. They do not alter ABS source values or attempt to “clean” confidentiality adjustments.

In [ ]:
g04 = workbook['G04']
g18 = workbook['G18']
g23 = workbook['G23']
g27 = workbook['G27']

# G04 values use the Persons column for ages 65 to 100+.
age_counts = {
    '65-74 years': g04.cell(row=21, column=14).value + g04.cell(row=27, column=14).value,
    '75-84 years': g04.cell(row=33, column=14).value + g04.cell(row=34, column=14).value,
    '85 years and over': sum(g04.cell(row=row, column=14).value for row in range(35, 39)),
}
total_population = g04.cell(row=40, column=14).value
older_population_g04 = sum(age_counts.values())

# G18 and G23 provide Persons rows for the three older age bands.
assistance_count = sum(g18.cell(row=row, column=2).value for row in [51, 52, 53])
assistance_denominator = sum(g18.cell(row=row, column=5).value for row in [51, 52, 53])
volunteer_count = sum(g23.cell(row=row, column=2).value for row in [45, 46, 47])
volunteer_denominator = sum(g23.cell(row=row, column=5).value for row in [45, 46, 47])

# G27 Persons section: Lone person row and 65-74, 75-84, 85+ columns.
lone_person_count = sum(g27.cell(row=54, column=column).value for column in [8, 9, 10])
lone_person_denominator = sum(g27.cell(row=57, column=column).value for column in [8, 9, 10])

summary_values = {
    'total_population': total_population,
    '65_plus_count_g04': older_population_g04,
    '65_plus_share_total_population_pct': older_population_g04 / total_population * 100,
    '65_plus_assistance_count': assistance_count,
    '65_plus_assistance_pct': assistance_count / assistance_denominator * 100,
    '65_plus_volunteer_count': volunteer_count,
    '65_plus_volunteer_pct': volunteer_count / volunteer_denominator * 100,
    '65_plus_lone_person_count': lone_person_count,
    '65_plus_lone_person_pct': lone_person_count / lone_person_denominator * 100,
}
pd.Series(summary_values).round(2)

In [ ]:
insights_df = pd.DataFrame([
    ['Age distribution', '65-74 years', age_counts['65-74 years'], 'count of persons', '65-74 years (G04)', 'Provides age-group context for inclusive age-friendly design.', 'Validate varied needs with older adults across age groups.'],
    ['Age distribution', '75-84 years', age_counts['75-84 years'], 'count of persons', '75-84 years (G04)', 'Provides age-group context for inclusive age-friendly design.', 'Validate varied needs with older adults across age groups.'],
    ['Age distribution', '85 years and over', age_counts['85 years and over'], 'count of persons', '85 years and over (G04)', 'Provides age-group context for inclusive age-friendly design.', 'Validate varied needs with older adults across age groups.'],
    ['Age distribution', '65 years and over share of total population', older_population_g04 / total_population * 100, 'percent', '65+ / total persons (G04)', 'Provides scale context for the 65+ population; not individual profiling.', 'Use aggregate context only; do not profile individuals.'],
    ['Core activity need for assistance', '65 years and over with a need for assistance', assistance_count, 'count of persons', '65+ has need / total 65+ (G18)', 'Provides context for low-effort, clear interaction design considerations.', 'Consider readable, simple, low-effort interaction; validate with users.'],
    ['Core activity need for assistance', '65 years and over with a need for assistance', assistance_count / assistance_denominator * 100, 'percent', '65+ has need / total 65+ (G18)', 'Provides context for low-effort, clear interaction design considerations.', 'Consider readable, simple, low-effort interaction; validate with users.'],
    ['Voluntary work', '65 years and over who volunteered', volunteer_count, 'count of persons', '65+ volunteer / total 65+ (G23)', 'Provides context for community participation pathways.', 'Consider community and local-discovery pathways; validate interest directly.'],
    ['Voluntary work', '65 years and over who volunteered', volunteer_count / volunteer_denominator * 100, 'percent', '65+ volunteer / total 65+ (G23)', 'Provides context for community participation pathways.', 'Consider community and local-discovery pathways; validate interest directly.'],
    ['Relationship in household', '65 years and over in Lone person category', lone_person_count, 'count of persons', '65+ Lone person / total 65+ in occupied private dwellings (G27)', 'Provides context for not assuming household support is always available.', 'Avoid assuming another household member is available for support.'],
    ['Relationship in household', '65 years and over in Lone person category', lone_person_count / lone_person_denominator * 100, 'percent', '65+ Lone person / total 65+ in occupied private dwellings (G27)', 'Provides context for not assuming household support is always available.', 'Avoid assuming another household member is available for support.'],
], columns=['indicator', 'category', 'value', 'unit', 'calculation_basis', 'agetogether_relevance', 'design_consideration'])

insights_df['geography'] = GEOGRAPHY
insights_df['census_year'] = CENSUS_YEAR
insights_df['value'] = insights_df['value'].round(2)
insights_df

In [ ]:
numeric_values = pd.to_numeric(insights_df['value'], errors='coerce')
print(f'Missing selected output values: {numeric_values.isna().sum()}')
print(f'Duplicate logical indicator/unit records: {insights_df.duplicated(["indicator", "category", "unit"]).sum()}')
print(f'Numeric parsing failures: {numeric_values.isna().sum()}')
print(f'Negative or impossible selected values: {(numeric_values < 0).sum()}')
print(f'Geography consistency: {(insights_df["geography"] == GEOGRAPHY).all()}')
print(f'Census year consistency: {(insights_df["census_year"] == CENSUS_YEAR).all()}')
print(f'Age-band total check (G04): {sum(age_counts.values()) == older_population_g04}')
print('Note: related Census table totals can differ slightly because ABS applies confidentiality adjustments.')

## 4. Target Audience Descriptive Analysis

The following findings provide demographic context only. They do not establish loneliness, digital capability, trust, accessibility needs, safety, or individual product preference.

### 4.1 Evidence-based findings

1. **Age profile:** The selected City of Melbourne LGA profile reports a measurable 65+ resident population, with most of those recorded in the 65–74 age band. This provides context for including older adults in the design scope rather than assuming one uniform older-user group.
2. **Core activity assistance:** The selected G18 table reports people aged 65+ with a core activity need for assistance. This may support a design consideration for clear, low-effort and age-friendly interaction patterns; it does not establish that any individual user needs assistance or cannot use digital services.
3. **Voluntary work:** The selected G23 table reports older residents who volunteered for an organisation or group. This provides context for considering community participation and local-discovery pathways; it is not a popularity measure or evidence that users want any particular activity.
4. **Lone person category:** The selected G27 table records older residents in the Census Lone person relationship category within occupied private dwellings. This may support avoiding an assumption that every user has another household member available for support. It must not be interpreted as proof of loneliness or isolation.

## 5. Hindsight → Insight → Foresight

| Hindsight (2021 Census evidence) | Insight (careful interpretation) | Foresight (later validation) |
|---|---|---|
| The profile reports 65+ residents across several age bands. | Older adults are not one homogeneous group; age-friendly design should accommodate varied contexts. | Test interface needs and feature priorities with older adults across different age groups. |
| The profile reports 65+ people with a core activity need for assistance. | This may support simple navigation, readable content and low-effort interaction as design considerations. | Validate accessibility needs with users and accessibility testing; do not infer them from Census aggregates. |
| The profile includes older residents in the Lone person relationship category. | The product should not assume a household support person is always present. | Validate trusted-family and local-social features with people living in different household arrangements. |

## 6. Visualisations

The charts describe 2021 aggregate Census counts/percentages for Melbourne LGA only. They do not compare popularity, quality, or product suitability.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(age_counts.keys(), age_counts.values(), color='#2b6cb0')
ax.set_title('Melbourne LGA: population aged 65+ by age group (2021 Census)')
ax.set_ylabel('Count of persons')
ax.set_xlabel('Age group')
for index, value in enumerate(age_counts.values()):
    ax.text(index, value + 80, f'{value:,.0f}', ha='center')
plt.tight_layout()
plt.show()

In [ ]:
percentage_chart = pd.Series({
    '65+ share of total population\n(G04)': older_population_g04 / total_population * 100,
    '65+ with core activity\nneed for assistance (G18)': assistance_count / assistance_denominator * 100,
    '65+ volunteers\n(G23)': volunteer_count / volunteer_denominator * 100,
    '65+ Lone person category\n(G27)': lone_person_count / lone_person_denominator * 100,
})
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(percentage_chart.index, percentage_chart.values, color='#4a9d76')
ax.set_title('Selected AgeTogether demographic context indicators: Melbourne LGA (2021 Census)')
ax.set_ylabel('Percent')
ax.set_ylim(0, max(percentage_chart.values) + 8)
for bar, value in zip(bars, percentage_chart.values):
    ax.text(bar.get_x() + bar.get_width() / 2, value + 0.6, f'{value:.1f}%', ha='center')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 7. Product Data View and Export

agetogether_abs_2021_demographic_insights.csv is a concise analysis output. It contains only indicators used in this notebook, not person-level records. Its design_consideration field records cautious Iteration 1 design context, not a recommendation or a user profile.

In [ ]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
insights_df.drop(columns=['calculation_basis']).to_csv(OUTPUT_FILE, index=False)

reloaded_output = pd.read_csv(OUTPUT_FILE)
assert len(reloaded_output) == len(insights_df)
assert list(reloaded_output.columns) == [
    'indicator', 'category', 'value', 'unit', 'agetogether_relevance',
    'design_consideration', 'geography', 'census_year'
]
assert raw_checksum == EXPECTED_SHA256
print(f'Processed output: {OUTPUT_FILE}')
print(f'Output records: {len(reloaded_output)}')
print('Reproducibility validation passed: raw checksum stable and repository-relative paths used.')

## 8. Limitations and Open Data Rubric Alignment

### Limitations

- Census statistics describe population characteristics in 2021, not 2026 conditions.
- Aggregate LGA statistics do not describe individual AgeTogether users.
- Census results should not be interpreted causally.
- ABS confidentiality perturbation/small random adjustments may cause small total differences.
- Demographic indicators cannot prove loneliness, trust, digital capability, accessibility requirements or product preference.
- Later user research and usability testing remain necessary.

### FIT5120 Open Data contribution

Clearly open/open-licensed sources in this project are:

1. City of Melbourne Places / POI
2. City of Melbourne / DataVic Activities and Planned Works
3. ABS 2021 Census demographic data

SBS News RSS is documented separately as an additional structured external metadata source, not as open data. Dataset 4 contributes target-audience/domain insight rather than direct frontend records.